# MCTS + SAC vs. raw SAC — local experiment harness

Compares an MCTS-guided agent (using `RL models/soft_actor_critic.keras`
for the policy prior and value estimate) against the same SAC model
running raw greedy argmax, so you can quickly measure whether MCTS at
inference improves on the policy alone — and which configuration of MCTS
gives the biggest lift.

**Run locally** (no Colab boilerplate). The model file is loaded directly
from the repo's `RL models/` folder. A single match of 20 games at 50
simulations/move takes roughly 1–2 minutes on CPU.

The agent code lives in `src/mcts.py`. Edit the cells below to vary
`n_simulations`, `c_puct`, and `value_method`; the sweep cell at the
bottom does this for you.


In [ ]:
# Cell 1 — setup, load the SAC model, build the two agents.
import os, sys
from pathlib import Path

# Walk up to the repo root (the directory that contains src/ and RL models/).
here = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in [here, *here.parents] if (p / 'src' / 'game_engine.py').exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not find repo root from {here}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print(f"REPO_ROOT = {REPO_ROOT}")

# Quiet TF down.
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
import tensorflow as tf
tf.get_logger().setLevel('ERROR')
import numpy as np

from src import model_loader as ml
from src.eval import ModelAgent, play_match, format_result
from src.mcts import MCTSAgent

# Load the SAC model directly. Encoding is auto-detected from input shape:
#   (None, 6, 7, 2) -> 'B'   (None, 6, 7, 1) -> 'A'   (None, 42, 2) -> 'B_flat'
SAC_PATH = REPO_ROOT / 'RL models' / 'soft_actor_critic.keras'
keras_model = tf.keras.models.load_model(str(SAC_PATH), compile=False)
in_shape = tuple(keras_model.input.shape)
trailing = in_shape[-3:] if len(in_shape) >= 3 else None
if trailing == (6, 7, 2):
    encoding = 'B'
elif trailing == (6, 7, 1):
    encoding = 'A'
elif in_shape[-2:] == (42, 2):
    encoding = 'B_flat'
else:
    raise ValueError(f"Unrecognised input shape {in_shape}")

sac_wrapper = ml.ModelWrapper(keras_model, encoding, name='SAC')
print(f"Loaded SAC: input shape {in_shape}, encoding={encoding}")

# Raw SAC opponent — greedy argmax over the policy + tactical overrides ON
# (matching the configuration submitted to tournament).
sac_raw = ModelAgent(sac_wrapper, name='sac_raw', greedy=True, use_tactics=True)
print(f"\nReady. SAC raw agent: {sac_raw.name}")


In [ ]:
# Cell 2 — helper: build an MCTS agent with a given config and play it
# vs the raw SAC agent for n_games. Returns the eval.MatchResult and prints
# a formatted summary.
def run_mcts_match(
    n_simulations=100,
    c_puct=1.4,
    value_method='mean_q',
    use_tactics=True,
    n_games=20,
    random_init_moves=4,
    add_root_noise=False,
):
    mcts_agent = MCTSAgent(
        sac_wrapper,
        n_simulations=n_simulations,
        c_puct=c_puct,
        value_method=value_method,
        use_tactics=use_tactics,
        add_root_noise=add_root_noise,
    )
    print(f"Match: {mcts_agent.name}  vs  {sac_raw.name}")
    print(f"  n_games={n_games}, c_puct={c_puct}, "
          f"random_init={random_init_moves}, tactics={use_tactics}")
    result = play_match(
        mcts_agent, sac_raw,
        n_games=n_games,
        random_init_moves=random_init_moves,
        progress=True,
    )
    print(format_result(result))
    print()
    return result


In [ ]:
# Cell 3 — quick single-config check (~1–2 minutes on CPU).
# Verifies the wiring works and gives a first directional reading.
result = run_mcts_match(
    n_simulations=50,
    c_puct=1.4,
    value_method='mean_q',
    n_games=20,
)

# Interpretation:
#   - a_win_rate >> 50%  -> MCTS adds real value over raw SAC.
#   - a_win_rate ~ 50%   -> MCTS roughly equivalent at this n_simulations.
#   - a_win_rate < 50%   -> MCTS is hurting (likely n_simulations too low,
#                          or value_method/c_puct mismatched to the model).


In [ ]:
# Cell 4 — sweep configurations and rank them.
# This cell takes longer (~10–20 min on CPU). Reduce n_games or the
# config list if you want a faster pass.
import pandas as pd

CONFIGS = [
    dict(n_simulations=50,  c_puct=1.4, value_method='mean_q'),
    dict(n_simulations=100, c_puct=1.4, value_method='mean_q'),
    dict(n_simulations=200, c_puct=1.4, value_method='mean_q'),
    dict(n_simulations=100, c_puct=1.0, value_method='mean_q'),
    dict(n_simulations=100, c_puct=2.0, value_method='mean_q'),
    dict(n_simulations=100, c_puct=1.4, value_method='rollout'),
    dict(n_simulations=200, c_puct=1.4, value_method='rollout'),
]
N_GAMES_PER_CONFIG = 30  # bump to 50–100 if you want tighter CIs

rows = []
for cfg in CONFIGS:
    res = run_mcts_match(n_games=N_GAMES_PER_CONFIG, **cfg)
    rows.append({
        'n_sims':       cfg['n_simulations'],
        'c_puct':       cfg['c_puct'],
        'value':        cfg['value_method'],
        'mcts_wins':    res.a_wins,
        'sac_wins':     res.b_wins,
        'draws':        res.draws,
        'mcts_winrate': res.a_win_rate,
        'avg_len':      round(res.avg_length, 1),
    })

df = pd.DataFrame(rows).sort_values('mcts_winrate', ascending=False)
print('\n=== Sweep summary (sorted by MCTS win rate) ===')
print(df.to_string(index=False))


## Reading the results

- **Win rate above ~55%**: MCTS+SAC is meaningfully stronger than raw SAC
  at that configuration. The bigger the margin, the better-justified the
  AlphaZero recommendation in the report (your tournament loss was to
  agents using exactly this kind of inference-time search).
- **Win rate near 50%**: at this `n_simulations` budget the search isn't
  adding signal beyond the policy. Try doubling `n_simulations` or
  switching `value_method`.
- **Win rate below 50%**: MCTS is actively hurting. Most likely causes,
  in order: (1) `n_simulations` too low so PUCT explores noisy children
  uniformly; (2) `value_method='mean_q'` but the SAC Q heads are poorly
  calibrated for inference (try `'rollout'`); (3) `c_puct` too small
  (try 2.0).

## Things to try next

- **Add an MCTS submission.**  If a configuration consistently wins
  ≥65% of games against raw SAC, that's the version to submit in any
  rematch. The same `MCTSAgent` class plugs into the live-match
  notebook via the `select_move(board, player)` interface.
- **Vary `n_simulations` higher** if you have GPU. 400 simulations/move
  is roughly the strength regime AlphaZero used at "casual" play; 1600+
  is tournament strength.
- **Disable `use_tactics`** to see the *pure* MCTS vs *pure* policy
  contrast. With tactics on, the immediate-win/block layer fires before
  MCTS in many positions, so part of the agent's strength comes from
  the heuristic rather than the search.
